# 4장 Seaborn 기초: 데이터와 차트 연결하기

『AI 시대의 파이썬 데이터 시각화』 예제 노트북입니다. 셀 첫 줄의 번호(P·C·B)는 책 지면의 코드 번호와 같습니다. 위에서부터 차례로 실행합니다.

## 준비

새 노트북이면 이 장의 코드보다 먼저 아래 준비 셀을 실행합니다(책의 앞 장과 같은 셀).

**코랩이면 이 셀(P02)을, 내 컴퓨터면 다음 셀(P03)을 실행합니다.** 둘 다 실행하지 않습니다.

In [ ]:
# P02 | 한글 글꼴(코랩): 나눔글꼴을 설치하고 Matplotlib에 등록합니다.
!apt-get -qq update
!apt-get -qq install -y fonts-nanum

import matplotlib.pyplot as plt
from matplotlib import font_manager

for path in font_manager.findSystemFonts():
    if 'Nanum' in path:
        font_manager.fontManager.addfont(path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

**내 컴퓨터(Mac·Windows)면 이 셀을 실행합니다.** Windows는 글꼴 이름을 `'Malgun Gothic'`으로 바꿉니다.

In [ ]:
# P03 | 한글 글꼴(Mac·Windows): 그래프를 그리기 전에 한 번 실행합니다.
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'AppleGothic'   # Windows는 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False    # 음수 기호가 깨지지 않게

## 4.1 표의 열과 차트 연결

In [ ]:
# P04 | Seaborn 준비: 표와 차트에 쓸 라이브러리를 불러옵니다.
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

print('Seaborn:', sns.__version__)   # errorbar= 인자는 0.12부터

In [ ]:
# C01 | 공통 데이터: A지점 1~4월 실적을 표로 만듭니다.
months = [1, 2, 3, 4]
sales = [100, 120, 150, 130]        # A지점 매출 (백만원)
profit = [10, 15, 25, 20]           # A지점 영업이익 (백만원)
sales_b = [70, 90, 100, 120]        # B지점 매출 (백만원)
branch_colors = {'A': '#0072B2', 'B': '#D55E00'}   # 지점 색

df = pd.DataFrame({'month': months, 'sales': sales, 'profit': profit})
print(df)
print('행과 열 개수:', df.shape)

In [ ]:
# C02 | 표와 열 이름으로 그리기: data·x·y·ax를 넘깁니다.
fig, ax = plt.subplots(figsize=(6, 2.8))
sns.lineplot(data=df, x='month', y='sales', ax=ax,
             marker='o', color=branch_colors['A'])
ax.set(title='A지점 1~4월 매출', xlabel='월',
       ylabel='매출 (백만원)', ylim=(0, 160))
ax.set_xticks(months)               # 눈금을 1~4월에만
fig.tight_layout()
plt.show()

### 직접 해 보기

**W4-01** C02를 A지점 영업이익 선으로 바꿉니다.

1. `y='profit'`으로 바꿉니다. 색과 마커는 자유롭게 정합니다.
2. 제목을 `A지점 1~4월 영업이익`, Y축 라벨을 `영업이익 (백만원)`으로 바꿉니다.
3. Y축 범위를 0~30으로 정합니다.
4. 영업이익이 가장 높은 달과 그 값을 그림에서 읽어 적습니다.

**완료 기준**: 열·제목·Y축 라벨·범위 네 가지가 모두 영업이익에 맞고, 가장 높은 달과 값을 단위와 함께 적었다.

In [ ]:
# W4-01 — 여기에서 직접 해 봅니다.

In [ ]:
# C03 | 넓은 형식: 지점마다 열이 하나인 표입니다.
wide = pd.DataFrame({'month': months, 'A': sales, 'B': sales_b})
print(wide)

In [ ]:
# C04 | 긴 형식: melt()로 지점을 열의 값으로 옮깁니다.
long = wide.melt(id_vars='month', value_vars=['A', 'B'],
                 var_name='branch', value_name='sales')
print(long)
print('값 보존 여부:', list(long['sales']) == sales + sales_b)
b_april = long.loc[(long['month'] == 4) & (long['branch'] == 'B'), 'sales']
print('B지점 4월 매출:', b_april.iloc[0])

## 4.2 색상 그룹과 요약값

In [ ]:
# C05 | hue: branch 열의 값으로 선을 나눕니다.
fig, ax = plt.subplots(figsize=(6, 2.8))
sns.lineplot(data=long, x='month', y='sales', hue='branch',
             style='branch', hue_order=['A', 'B'],
             palette=branch_colors, marker='o',
             errorbar=None, ax=ax)
ax.set(title='1~4월 지점별 매출', xlabel='월',
       ylabel='매출 (백만원)', ylim=(0, 160))
ax.set_xticks(months)
ax.legend(title='지점', loc='lower right')
fig.tight_layout()
plt.show()

In [ ]:
# C06 | hue가 없으면: 같은 달의 두 지점 값을 평균합니다.
fig, ax = plt.subplots(figsize=(6, 2.8))
sns.lineplot(data=long, x='month', y='sales', ax=ax,
             marker='o', errorbar=None, color='#666666')
ax.set(title='월별 두 지점의 평균 매출', xlabel='월',
       ylabel='매출 (백만원)', ylim=(0, 160))
ax.set_xticks(months)
fig.tight_layout()
plt.show()

### 직접 해 보기

**W4-02** C06의 인자를 바꿔 선이 무엇을 보여 주는지 확인합니다.

1. `errorbar=None`을 지우고 실행합니다. 무엇이 더 그려지는지, 선의 높이는 바뀌는지 봅니다.
2. `hue='branch'`를 더해 실행하고 그림 4-4와 같은 위치에 두 선이 생기는지 확인합니다. 색은 기본 색으로 바뀝니다.
3. 그림 4-5에서 3월과 4월의 점이 모두 125로 같은 까닭을 두 지점의 매출 변화로 설명합니다.

**완료 기준**: 오차 표시가 선의 높이(평균)를 바꾸지 않는다는 것, `hue`를 더하면 월·지점마다 원래 값이 그려진다는 것, 3·4월 평균이 같은 까닭을 각각 한 문장으로 설명했다.

In [ ]:
# W4-02 — 여기에서 직접 해 봅니다.

In [ ]:
# C07 | 반복 관측: 같은 그룹에 행이 둘이면 hue가 있어도 평균합니다.
extra = pd.DataFrame({'month': [1], 'branch': ['A'], 'sales': [110]})
repeated = pd.concat([long, extra], ignore_index=True)   # long은 그대로
a_jan = (repeated['month'] == 1) & (repeated['branch'] == 'A')
print(repeated[a_jan])
print('A지점 1월 선의 값:', repeated.loc[a_jan, 'sales'].mean())

In [ ]:
# C08 | 행 개수: countplot()은 값마다 행이 몇 개인지 막대로 셉니다.
fig, ax = plt.subplots(figsize=(6, 2.4))
sns.countplot(data=repeated, x='branch', order=['A', 'B'], hue='branch',
              palette=branch_colors, legend=False, ax=ax)
ax.set(title='지점별 행 개수 (반복 관측을 더한 사본)',
       xlabel='지점', ylabel='행 개수', ylim=(0, 6))
for bars in ax.containers:          # 지점마다 막대 묶음이 하나씩
    ax.bar_label(bars, padding=4)   # 막대 위에 개수를 적습니다
fig.tight_layout()
plt.show()

In [ ]:
# C09 | 막대의 요약값: barplot()은 지점마다 네 달의 평균을 그립니다.
fig, ax = plt.subplots(figsize=(6, 2.8))
sns.barplot(data=long, x='branch', y='sales', order=['A', 'B'],
            hue='branch', palette=branch_colors, legend=False,
            estimator='mean', errorbar=None, ax=ax)
ax.set(title='지점별 월평균 매출 (1~4월)', xlabel='지점',
       ylabel='월평균 매출 (백만원)', ylim=(0, 160))
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.0f', padding=4)
fig.tight_layout()
plt.show()

In [ ]:
# C10 | 합계 표: 합계를 물으면 표부터 지점별로 더합니다.
totals = long.groupby('branch', as_index=False)['sales'].sum()
print(totals)

### 직접 해 보기

**W4-03** C10의 `totals`로 지점별 합계 막대를 그립니다.

1. C09를 바탕으로 `data=totals`로 바꾸고, 지점마다 한 행이라 평균할 것이 없으므로 `estimator='mean'`은 지웁니다.
2. 제목을 `지점별 1~4월 합계 매출`, Y축 라벨을 `1~4월 합계 매출 (백만원)`으로 바꿉니다.
3. Y축 범위를 0~600으로 정하고 막대 위에 값을 적습니다.
4. 그림 4-7의 막대와 무엇이 다른지 한 문장으로 적습니다.

**완료 기준**: 막대가 C10 합계와 같고 제목·Y축 라벨이 합계를 밝히며, 두 그림의 차이를 숫자와 단위로 설명했다.

In [ ]:
# W4-03 — 여기에서 직접 해 봅니다.

In [ ]:
# C11 | 묶음 막대 두 방법: 위치를 직접 계산하거나 hue로 나눕니다.
fig, axes = plt.subplots(1, 2, sharey=True, figsize=(6, 2.8))
width = 0.4
axes[0].bar([m - width / 2 for m in months], sales, width=width,
            color=branch_colors['A'], label='A')
axes[0].bar([m + width / 2 for m in months], sales_b, width=width,
            color=branch_colors['B'], hatch='//', label='B',
            edgecolor='black', linewidth=0)       # 빗금 선 색
axes[0].set_xticks(months)
axes[0].set(title='Matplotlib: 위치 계산', xlabel='월',
            ylabel='매출 (백만원)', ylim=(0, 200))
axes[0].legend(title='지점')
sns.barplot(data=long, x='month', y='sales', hue='branch',
            palette=branch_colors, errorbar=None, ax=axes[1])
for bar in axes[1].containers[1]:     # B지점 막대 묶음
    bar.set(hatch='//', edgecolor='black', linewidth=0)
axes[1].set(title='Seaborn: hue', xlabel='월')
axes[1].legend(axes[1].containers, ['A', 'B'], title='지점')
fig.tight_layout()
plt.show()

## 4.3 차트 배치와 결과 저장

In [ ]:
# C12 | 두 차트의 골격: 함수마다 그릴 Axes를 ax=로 정합니다.
fig, axes = plt.subplots(1, 2, figsize=(6, 3.0))
left = sns.lineplot(data=long, x='month', y='sales', hue='branch',
                    style='branch', hue_order=['A', 'B'],
                    palette=branch_colors, marker='o',
                    errorbar=None, ax=axes[0])      # 왼쪽: 월별 변화
sns.barplot(data=long, x='branch', y='sales', order=['A', 'B'],
            hue='branch', palette=branch_colors, legend=False,
            estimator='mean', errorbar=None, ax=axes[1])  # 오른쪽: 월평균
print('반환값이 axes[0]인가?', left is axes[0])
fig.tight_layout()
plt.show()

In [ ]:
# C13 | 완성: Y축을 공유하고 제목과 단위를 답니다.
fig, axes = plt.subplots(1, 2, sharey=True, figsize=(6, 3.0))
sns.lineplot(data=long, x='month', y='sales', hue='branch',
             style='branch', hue_order=['A', 'B'],
             palette=branch_colors, marker='o',
             errorbar=None, ax=axes[0])
axes[0].set(title='월별 매출 (원래 값)', xlabel='월',
            ylabel='매출 (백만원)', ylim=(0, 160))
axes[0].set_xticks(months)
axes[0].legend(title='지점', loc='lower right')

sns.barplot(data=long, x='branch', y='sales', order=['A', 'B'],
            hue='branch', palette=branch_colors, legend=False,
            estimator='mean', errorbar=None, ax=axes[1])
for bar in axes[1].containers[1]:     # B지점 막대에 빗금(C11)
    bar.set(hatch='//', edgecolor='black', linewidth=0)
axes[1].set(title='월평균 매출 (네 달 평균)', xlabel='지점')
# Seaborn이 숨겨 둔 오른쪽 Y축 라벨을 다시 보이게 합니다.
axes[1].set_ylabel('월평균 매출 (백만원)', visible=True)
for bars in axes[1].containers:
    axes[1].bar_label(bars, fmt='%.0f', padding=4)
fig.suptitle('1~4월 지점별 매출 비교')
fig.tight_layout()
plt.show()

In [ ]:
# C13B | 검산: 그림의 막대 높이를 표에서 계산한 평균과 대조합니다.
heights = [round(bar.get_height()) for bar in axes[1].patches]
print('막대 높이:', heights)
print('표의 월평균:', long.groupby('branch')['sales'].mean().tolist())

### 직접 해 보기

**W4-04** C13의 그림을 저장해 검수한 뒤, 관찰을 두 문장으로 씁니다.

1. 2장 C05처럼 `ch04_output` 폴더를 만들고 그림을 `W4-04.png`로 저장합니다. `dpi=150`과 `bbox_inches='tight'`를 씁니다.
2. 2장 C06처럼 파일을 다시 열어 제목, 월 1~4, A·B 범례, 단위, 막대 값 125·95, 잘린 글자가 없는지 확인합니다.
3. 3월에서 4월까지 두 지점 매출의 변화를 숫자와 단위로 한 문장에 씁니다.
4. 1~4월 월평균 매출의 차이를 한 문장으로 씁니다. 변화의 원인은 단정하지 않습니다.

**완료 기준**: 저장한 파일에서 여섯 가지를 확인했고, 두 문장에 기간·지점·숫자·단위가 들어 있다.

In [ ]:
# W4-04 — 여기에서 직접 해 봅니다.

## 4.4 해석과 점검

## 직접 해 보기 풀이

먼저 위의 연습 칸에서 직접 해 본 뒤에 봅니다. 풀이는 한 가지 예이고, 완료 기준을 만족하면 다른 코드도 됩니다.

### W4-01

In [ ]:
# W4-01 | 풀이: C02를 A지점 영업이익 선으로 바꿉니다.
fig, ax = plt.subplots(figsize=(6, 2.8))
sns.lineplot(data=df, x='month', y='profit', ax=ax,
             marker='s', color=branch_colors['A'])
ax.set(title='A지점 1~4월 영업이익', xlabel='월',
       ylabel='영업이익 (백만원)', ylim=(0, 30))
ax.set_xticks(months)
fig.tight_layout()
plt.show()

In [ ]:
# 4. 가장 높은 달과 값(그림에서 읽은 값과 대조)
print(df.loc[df['profit'] == df['profit'].max(), ['month', 'profit']])
assert ax.get_ylim() == (0, 30) and ax.get_title() == 'A지점 1~4월 영업이익'
# 답(예): 3월에 25백만원으로 가장 높다.

### W4-02

In [ ]:
# W4-02 | 풀이: C06에서 errorbar=None을 지우고(왼쪽), hue='branch'를 더해(오른쪽) 견줍니다.
fig, axes = plt.subplots(1, 2, sharey=True, figsize=(6, 2.8))
sns.lineplot(data=long, x='month', y='sales', ax=axes[0],
             marker='o', color='#666666')                     # 1. errorbar=None을 지움
axes[0].set(title='오차 표시를 켠 평균', xlabel='월',
            ylabel='매출 (백만원)', ylim=(0, 160))
sns.lineplot(data=long, x='month', y='sales', hue='branch',
             palette=branch_colors, marker='o', errorbar=None, ax=axes[1])   # 2. hue를 더함
axes[1].set(title='hue를 더한 선', xlabel='월')
axes[1].legend(title='지점')
for ax in axes:
    ax.set_xticks(months)
fig.tight_layout()
plt.show()

In [ ]:
# 선의 높이는 평균 그대로이고, 3·4월 평균은 모두 125
print('왼쪽 선의 높이:', list(axes[0].lines[0].get_ydata()))
print(long.groupby('month')['sales'].mean())
print(long.loc[long['month'] >= 3])
assert list(axes[0].lines[0].get_ydata()) == [85, 105, 125, 125]
# 답(예): 오차 표시를 켜면 평균 둘레에 띠가 더 그려질 뿐 선의 높이(평균)는 85·105·125·125백만원 그대로다.
#   hue='branch'를 더하면 월·지점마다 원래 값이 그려져 그림 4-4와 같은 두 선이 된다.
#   3월(A 150, B 100)에서 4월(A 130, B 120)로 A가 20 줄고 B가 20 늘어 두 달의 평균이 모두 125백만원이다.

### W4-03

In [ ]:
# W4-03 | 풀이: C10의 totals로 지점별 합계 막대를 그립니다.
fig, ax = plt.subplots(figsize=(6, 2.8))
sns.barplot(data=totals, x='branch', y='sales', order=['A', 'B'],
            hue='branch', palette=branch_colors, legend=False,
            errorbar=None, ax=ax)                 # estimator='mean'은 지움(지점마다 한 행)
ax.set(title='지점별 1~4월 합계 매출', xlabel='지점',
       ylabel='1~4월 합계 매출 (백만원)', ylim=(0, 600))
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.0f', padding=4)
fig.tight_layout()
plt.show()

In [ ]:
heights = [round(bar.get_height()) for bar in ax.patches]
print('막대 높이:', heights)
assert heights == [500, 380]
# 차이(예): 그림 4-7의 막대는 네 달의 월평균(A 125, B 95백만원)이고, 이 막대는 네 달 합계(A 500, B 380백만원)다.

### W4-04

In [ ]:
# W4-04 | 풀이: C13의 그림을 저장하고 다시 열어 검수합니다.
from pathlib import Path

output_dir = Path('ch04_output')
output_dir.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, sharey=True, figsize=(6, 3.0))
sns.lineplot(data=long, x='month', y='sales', hue='branch',
             style='branch', hue_order=['A', 'B'],
             palette=branch_colors, marker='o',
             errorbar=None, ax=axes[0])
axes[0].set(title='월별 매출 (원래 값)', xlabel='월',
            ylabel='매출 (백만원)', ylim=(0, 160))
axes[0].set_xticks(months)
axes[0].legend(title='지점', loc='lower right')

sns.barplot(data=long, x='branch', y='sales', order=['A', 'B'],
            hue='branch', palette=branch_colors, legend=False,
            estimator='mean', errorbar=None, ax=axes[1])
for bar in axes[1].containers[1]:     # B지점 막대에 빗금(C11)
    bar.set_hatch('//')
axes[1].set(title='월평균 매출 (네 달 평균)', xlabel='지점')
# Seaborn이 숨겨 둔 오른쪽 Y축 라벨을 다시 보이게 합니다.
axes[1].set_ylabel('월평균 매출 (백만원)', visible=True)
for bars in axes[1].containers:
    axes[1].bar_label(bars, fmt='%.0f', padding=4)
fig.suptitle('1~4월 지점별 매출 비교')
fig.tight_layout()
fig.savefig(output_dir / 'W4-04.png', dpi=150, bbox_inches='tight')   # 1. 저장
plt.show()

In [ ]:
# 2. 다시 열어 제목, 월 1~4, A·B 범례, 단위, 막대 값 125·95, 잘린 글자를 봅니다.
from IPython.display import Image, display

display(Image(filename='ch04_output/W4-04.png', width=600))

In [ ]:
# 3·4. 문장에 쓸 값
print(wide.loc[wide['month'] >= 3])
print(long.groupby('branch')['sales'].mean())
# 관찰(예): 3월에서 4월까지 A지점 매출은 150에서 130백만원으로 20 줄고, B지점은 100에서 120백만원으로 20 늘었다.
# 관찰(예): 1~4월 월평균 매출은 A지점 125, B지점 95백만원으로 A지점이 30백만원 많다.